![OneHealth DataSpace](https://bigdata.dataspace.cesga.es/static/images/public/imagotipo.png)

> **⚠️ ADVERTENCIA**: actualmente a plataforma está en fase de proba. Agradecemos que nos fagades chegar calquera comentario a onehealth@cesga.es

Neste notebook imos ver como analizar a profundidade de captura do bacalao a partir dos datos de caputras de DIVERSIMAR dun xeito totalmente interactivo.

In [ ]:
from pyspark.sql import functions as F
import onehealth

capturas = onehealth.load("SS4jXDXK2DgTDLE")
# ============================================================
# 1. PREPARAR LOS DATOS
# ============================================================

df = capturas.select(
    "spanish_fao_name",
    F.col("depth").cast("double").alias("depth"),
    F.col("individuals").cast("double").alias("individuals")
).filter(
    F.col("spanish_fao_name").isNotNull() &
    F.col("depth").isNotNull() &
    F.col("individuals").isNotNull()
)


# ============================================================
# 2. CREAR INTERVALOS DE PROFUNDIDAD DE 50 m
# ============================================================

df_binned = df.withColumn(
    "x",
    (F.floor(F.col("depth") / 50) * 50 + 25).cast("double")
)


# ============================================================
# 3. SUMAR CAPTURAS POR ESPECIE Y PROFUNDIDAD
# ============================================================

agrupado = (
    df_binned
    .groupBy("spanish_fao_name", "x")
    .agg(
        F.sum("individuals").alias("top")
    )
    .orderBy("spanish_fao_name", "x")
)


# Pasamos el resultado a Pandas para trabajar con Bokeh
pdf = agrupado.toPandas()


# ============================================================
# 4. LISTADO DE ESPECIES
# ============================================================

listado_especies = sorted(
    pdf["spanish_fao_name"]
    .dropna()
    .unique()
    .tolist()
)

print("Número de especies:", len(listado_especies))


# ============================================================
# 5. CREAR DICCIONARIO 'datos'
# ============================================================

datos = {}

for especie in listado_especies:

    temp = pdf[
        pdf["spanish_fao_name"] == especie
    ].sort_values("x")

    datos[especie] = {
        "x": temp["x"].tolist(),
        "top": temp["top"].tolist(),
        "color": ["steelblue"] * len(temp)
    }


In [ ]:
from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import (
    ColumnDataSource,
    Select,
    CustomJS,
    FixedTicker,
    NumeralTickFormatter,
    HoverTool
)
from bokeh.layouts import column

output_notebook()


# ============================================================
# 1. LISTADO DE ESPECIES
# ============================================================

listado_especies = sorted(
    pdf["spanish_fao_name"]
    .dropna()
    .unique()
    .tolist()
)


# ============================================================
# 2. CREAR UNA FUENTE DE DATOS PARA CADA ESPECIE
# ============================================================

sources = {}

for especie in listado_especies:

    temp = (
        pdf[pdf["spanish_fao_name"] == especie]
        .sort_values("x")
    )

    sources[especie] = ColumnDataSource(
        data={
            "x": temp["x"].tolist(),
            "top": temp["top"].tolist()
        }
    )


# ============================================================
# 3. CREAR "TODAS LAS ESPECIES"
# ============================================================

todas = (
    pdf.groupby("x", as_index=False)["top"]
    .sum()
    .sort_values("x")
)

sources["Todas las especies"] = ColumnDataSource(
    data={
        "x": todas["x"].tolist(),
        "top": todas["top"].tolist()
    }
)


# ============================================================
# 4. ESPECIE INICIAL
# ============================================================

especie_inicial = "Pulpo blanco"

source = ColumnDataSource(
    data=dict(sources[especie_inicial].data)
)


# ============================================================
# 5. CREAR GRÁFICO
# ============================================================

p = figure(
    title=f"Profundidad de captura - {especie_inicial}",
    x_axis_label="Profundidad (m)",
    y_axis_label="Número de individuos",
    height=600,
    width=950,
    toolbar_location="right"
)


p.vbar(
    x="x",
    top="top",
    bottom=0,
    width=45,
    source=source,
    line_color="black",
    fill_color="steelblue",
    alpha=0.7
)


# ============================================================
# 6. EJES
# ============================================================

p.xaxis.ticker = FixedTicker(
    ticks=list(range(0, 1551, 100))
)

p.xaxis.formatter = NumeralTickFormatter(format="0")
p.yaxis.formatter = NumeralTickFormatter(format="0,0")

p.xaxis.major_label_text_font_size = "11pt"
p.yaxis.major_label_text_font_size = "11pt"

p.min_border_bottom = 70
p.min_border_left = 80


# ============================================================
# 7. HOVER
# ============================================================

hover = HoverTool(
    tooltips=[
        ("Profundidad", "@x{0} m"),
        ("Individuos", "@top{0,0}")
    ]
)

p.add_tools(hover)


# ============================================================
# 8. SELECTOR
# ============================================================

opciones = ["Todas las especies"] + listado_especies

select = Select(
    title="Especie:",
    value=especie_inicial,
    options=opciones,
    width=300
)


# ============================================================
# 9. CALLBACK JAVASCRIPT
# ============================================================

callback = CustomJS(
    args=dict(
        source=source,
        sources=sources,
        p=p
    ),
    code="""

        const especie = cb_obj.value;

        const nueva_fuente = sources[especie];

        source.data = {...nueva_fuente.data};
        source.change.emit();

        if (especie === "Todas las especies") {

            p.title.text =
                "Profundidad de captura - Todas las especies";

        } else {

            p.title.text =
                "Profundidad de captura - " + especie;

        }
    """
)

select.js_on_change("value", callback)


# ============================================================
# 10. MOSTRAR
# ============================================================

show(column(select, p))